[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/02_input_data_prep/02_data_prep.ipynb)


# 02 — 입력 데이터 준비

> 본문: [`02_input_data_prep.md`](02_input_data_prep.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 `data/` 의 **실제 BoltzGen 실행 결과**에서 계산합니다(임의값 아님).
> **분석 셀 실행 4초.**

> 런타임 변경 없이 그대로 실행하세요.

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 노트북은 **Colab과 로컬 모두**에서 동작합니다.

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`02_input_data_prep`) 폴더로 자동 이동한 뒤, `data/` 의 실제 결과로 실습합니다.
- **로컬**: 이 노트북을 `02_input_data_prep/` 폴더 안에서 열었다면 클론 없이 그대로 진행됩니다.

> 런타임은 **기본값 그대로** 두면 됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "02_input_data_prep"
PIP_PKGS = "gemmi"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# 필요한 라이브러리 확보: import 안 되는 것만 설치(Colab 새 런타임/로컬 모두 자동 대응)
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# ── 내가 만든 결과 우선, 없으면 레퍼런스 ──────────────────────────────────────
#   MY  : 이 노트북에서 내가 직접 돌려 만든 산출물
#   REF : 저장소에 커밋된 레퍼런스 결과(대조군 · 실행을 건너뛰어도 실습이 이어지도록)
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """산출물 하나를 찾는다 — 내가 만든 my_run/ 트리를 먼저 뒤지고, 없으면 레퍼런스 폴더에서.

    파일명 규칙이 달라도(내 실행 rank1_*.cif / 레퍼런스 rank001_*.cif,
    final_<budget>_designs / final_designs) 같은 글롭으로 잡히도록 설계했습니다.
    """
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 이미 설계를 돌렸다면 그 결과를 이어받는다(이 챕터에서 다시 안 돌려도 됨).

    내 my_run/ 에 결과가 있으면 그대로 쓰고, 없으면 인자로 준 순서대로 앞 챕터를 찾는다.
    아무 데도 없으면 MY 를 그대로 둬서 find_one() 이 레퍼런스로 폴백하게 한다.
    """
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 내가 만든 그림은 my_ 접두어로 저장 — 저장소에 커밋된 레퍼런스 그림을 덮어쓰지 않도록.
def my_fig(name):
    return f"my_{name}"

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd())

## 1) 타깃 구조 다운로드 (본문 2.3)
RCSB PDB 에서 타깃 좌표를 받습니다. 예시는 PD-L1 (`7uxq`).

In [ ]:
import urllib.request, pathlib
work = pathlib.Path("work"); work.mkdir(exist_ok=True)
pid = "7uxq"                                  # PD-L1
cif = work / f"{pid}.cif"
if not cif.exists():
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pid}.cif", cif)
print("다운로드:", cif, "|", cif.stat().st_size, "bytes")

## 2) 체인·entity 타입 확인 (gemmi)
타깃의 체인과 폴리머 종류(단백질/DNA/RNA)를 확인해 설계 명세 작성의 근거로 삼습니다.

In [ ]:
import gemmi
st = gemmi.read_structure(str(cif))
print("title:", st.name)
for ch in st[0]:
    kinds = {r.name for r in ch}
    poly = ch.get_polymer()
    ptype = poly.check_polymer_type() if len(poly) else "-"
    print(f"  chain {ch.name}: {len(ch):4d} residues | type={ptype} | 첫 잔기 {ch[0].name}")

## 3) entity 5종 — 설계 명세의 구성요소 (본문 2.2)
BoltzGen 명세는 **무엇을 만들지(설계 대상)** 와 **무엇에 붙일지(타깃)** 를 entity 로 기술합니다.

In [ ]:
# (a) 설계 대상: 단백질 — 길이 범위 / 고정 길이 / 고정 서열
print("- protein: { id: B, sequence: 80..140 }     # 80~140aa 중 무작위 길이로 설계")
print("- protein: { id: B, sequence: 120 }          # 고정 120aa")
print("- protein: { id: B, sequence: MKLVAA... }     # 서열 고정(재설계/스캐폴드)")
print()
# (b) 핵산 타깃
print("- dna: { id: D, sequence: ATGCGT }")
print("- rna: { id: R, sequence: AUGCGU }")
print()
# (c) 소분자 타깃: CCD 코드 또는 SMILES
print('- ligand: { id: L, ccd: ATP }                 # PDB 화학성분 사전 3글자 코드')
print('- ligand: { id: L, smiles: "CC(=O)Oc1ccccc1C(=O)O" }   # 아스피린(CCD에 없을 때)')

## 4) file entity — 타깃 정제·결합부위 지정 (본문 2.4~2.5)
실제 구조 파일을 타깃으로 쓸 때의 핵심 옵션을 한 명세로 모읍니다.
- `include` 쓸 체인, `exclude` 유연한 루프/무질서 영역 제거
- `binding_types` 결합 유도 활성부위, `not_binding` 그 외 표면 억제
- `reset_res_index` 잔기번호 재정렬, `structure_groups` 가시성 그룹

In [ ]:
spec = """entities:
  - protein: { id: B, sequence: 80..140 }       # 설계할 바인더
  - file:
      path: work/7uxq.cif
      include:       [ { chain: { id: A } } ]
      exclude:       [ { chain: { id: A, res_index: 45..55 } } ]   # 유연 루프 제거(예)
      binding_types: [ { chain: { id: A, binding: 50..70 } } ]     # 결합 유도 부위
      reset_res_index: [ { chain: { id: A } } ]
      structure_groups: "all"
"""
open("work/my_spec.yaml", "w").write(spec)
print(spec)

## 5) 핵산(DNA/RNA) 타깃 — 자동 인식 (본문 2.7)
CIF 안의 DNA/RNA 체인은 BoltzGen 이 자동 인식하므로, 단백질 타깃과 **동일하게** `include` 만 하면 됩니다.

In [ ]:
dna_spec = """entities:
  - protein: { id: G, sequence: 40..120 }                 # DNA 결합 단백질 설계
  - file:
      path: example/denovo_zinc_finger_against_dna/zf.cif
      include: [ { chain: { id: C1 } }, { chain: { id: B1 } } ]   # DNA 이중가닥 두 체인
      structure_groups: "all"
"""
print(dna_spec)

## 6) 품질 체크 — `boltzgen check` (본문 2.8)
설계 명세가 올바른지 실행 전에 검증합니다. `Total designed residues: N` 과 시각화 CIF 가 나오면 정상입니다.
아래 셀이 **boltzgen 을 설치하고 예제 명세로 실제 검증까지** 돌립니다.
Colab 첫 실행은 설치에 몇 분 걸립니다(GPU 없이 CPU 로도 검증됩니다 — `check` 는 GPU 가 필요 없어요). 설치 상세는 Ch.03.

In [ ]:
import shutil, subprocess, pathlib, sys

# Colab: boltzgen 이 없으면 설치합니다(check 는 CPU 로도 동작 — GPU 불필요).
if not shutil.which("boltzgen"):
    _run(f'"{sys.executable}" -m pip -q install boltzgen')

# 예제 명세(1g13prot.yaml)는 pip 패키지에 없고 BoltzGen 소스 레포에만 있어 받아옵니다(있으면 건너뜀).
SRC = ADV_ROOT / ".boltzgen_src"
if not SRC.exists():
    _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')

spec = SRC / "example/vanilla_protein/1g13prot.yaml"
if shutil.which("boltzgen") and spec.exists():
    subprocess.run(["boltzgen", "check", str(spec)])
    print("\n→ 'Total designed residues: N' 과 1g13prot.cif(시각화 구조)가 나오면 명세가 정상입니다.")
    print("  아래 7) 셀이 이 구조를 코랩 화면에 바로 3D 로 띄웁니다(PyMOL 불필요).")
else:
    print("boltzgen 설치가 아직 안 됐습니다 — 이 셀을 한 번 더 실행하세요(설치 완료 후 자동으로 검증됩니다).")

## 7) 명세 구조를 코랩에서 바로 3D 로 보기 — PyMOL 불필요
방금 `boltzgen check` 가 만든 `1g13prot.cif` 를 코랩 셀 안에서 곧장 3D 로 돌려봅니다.
`py3Dmol` 이 3Dmol.js 뷰어를 노트북 출력에 심어줘서, 마우스로 **회전·확대**할 수 있어요.
N말단(파랑) → C말단(빨강) 무지개로 칠해, 타깃 사슬이 어떻게 접혀 있는지 한눈에 봅니다.

In [ ]:
# 코랩 셀 안에서 바로 3D — PyMOL 없이 (py3Dmol = 3Dmol.js 뷰어)
import importlib.util, pathlib
if importlib.util.find_spec("py3Dmol") is None:
    _run(f'"{sys.executable}" -m pip -q install py3Dmol')
import py3Dmol, gemmi

cif = pathlib.Path("1g13prot.cif")            # 위 6) boltzgen check 가 만든 시각화 구조
assert cif.exists(), "1g13prot.cif 가 없어요 — 위 6) 셀을 먼저 실행하세요."

# 3Dmol.js 는 PDB 를 가장 안정적으로 읽어요 → gemmi 로 CIF→PDB 변환(gemmi 는 0) 셀에서 설치됨)
pdb = gemmi.read_structure(str(cif)).make_pdb_string()

view = py3Dmol.view(width=760, height=540)
view.addModel(pdb, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})   # N말단 파랑 → C말단 빨강
view.setBackgroundColor("white")
view.zoomTo()
view.show()                                   # ← 코랩/주피터 출력에 인라인 3D 뷰어가 뜹니다